[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C07_ML_Foundations_Course/04_unsupervised_learning/04_unsupervised_learning.ipynb)

# 04 · 无监督学习：PCA、K-means、GMM/EM

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span>
在真实 **手写数字 optdigits**（8×8 图像）和 **UCI Wine** 上，从零写 PCA(SVD)、k-means、GMM-EM。

**你将完成：**
1. PCA via SVD：把 64 维数字降到 2 维，看解释方差比
2. k-means 从零 + inertia
3. cluster purity：用真实数字标签验证聚类质量
4. GMM 一维 EM：软分配（responsibilities）

> 数据：UCI optdigits（8×8 手写数字, 64 特征）+ UCI Wine（178 瓶, 13 化学特征, 3 品种）。

## 0 · 加载真实数据

In [ ]:
import os, urllib.request
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True)
CACHE=os.path.expanduser("~/.ml_foundations_data"); os.makedirs(CACHE, exist_ok=True)
def fetch(u,f):
    p=os.path.join(CACHE,f)
    if not os.path.exists(p): urllib.request.urlretrieve(u,p)
    return p

dig = pd.read_csv(fetch("https://archive.ics.uci.edu/ml/machine-learning-databases/optdigits/optdigits.tra","optdigits.tra"), header=None)
Xd = dig.iloc[:,:64].to_numpy(float); yd = dig.iloc[:,64].to_numpy(int)
print("手写数字 X:", Xd.shape, " 标签:", sorted(set(yd)))

wine = pd.read_csv(fetch("https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data","wine.data"), header=None)
Xw = wine.iloc[:,1:].to_numpy(float); yw = wine.iloc[:,0].to_numpy(int)-1
Xw = (Xw - Xw.mean(0))/Xw.std(0)   # 标准化（量纲差极大）
print("红酒 X:", Xw.shape, " 品种:", sorted(set(yw)))

## 1 · PCA via SVD（手写数字降维）

中心化 → SVD → 取前 k 个右奇异向量作主成分。解释方差比 = σ²ₖ / Σσ²。

In [ ]:
def pca(X, k):
    Xc = X - X.mean(0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    evr = S**2 / (S**2).sum()
    Z = Xc @ Vt[:k].T            # 投影到前 k 个主成分
    return Z, Vt[:k], evr

Z, comps, evr = pca(Xd, 2)
print("前 2 主成分解释方差比:", evr[:2].round(3), " 累计:", evr[:2].sum().round(3))
print("前 10 主成分累计解释方差比:", evr[:10].sum().round(3), "(64维数字大部分信息在前10维)")
# 2D 投影后不同数字是否分开？用类内/类间距离粗看
from itertools import combinations
centers = np.array([Z[yd==d].mean(0) for d in range(10)])
print("2D 投影后 10 个数字的簇中心两两可分（类间距 >> 类内散度）:",
      np.linalg.norm(centers[0]-centers[1]) > Z[yd==0].std())

## 2 · k-means 从零（手写数字）

Lloyd 迭代：分配到最近中心 → 中心取簇均值。记录 inertia（簇内平方和）。

In [ ]:
def kmeans(X, k, n_iter=50, seed=0):
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), k, replace=False)].copy()
    for _ in range(n_iter):
        d = ((X[:,None,:]-centers[None,:,:])**2).sum(2)   # (n,k)
        labels = d.argmin(1)
        new = np.array([X[labels==j].mean(0) if (labels==j).any() else centers[j] for j in range(k)])
        if np.allclose(new, centers): break
        centers = new
    inertia = ((X - centers[labels])**2).sum()
    return labels, centers, inertia

lab, cen, inertia = kmeans(Xd, 10, seed=0)
print(f"k-means (k=10) inertia = {inertia:,.0f}")
# 多次初始化取最优（k-means 只到局部最优）
best = min((kmeans(Xd,10,seed=s)[2] for s in range(5)))
print(f"5 次随机初始化最优 inertia = {best:,.0f}  (说明初始化影响结果)")

## 3 · cluster purity（用真实标签验证）

每个簇取其中最多的真实数字当"预测"，正确数 / 总数 = purity。这是有外部标签时的聚类评估。

In [ ]:
def purity(labels, y_true):
    total = 0
    for c in np.unique(labels):
        mask = labels==c
        total += np.bincount(y_true[mask]).max()
    return total/len(y_true)

p = purity(lab, yd)
print(f"k-means 聚类 purity = {p:.3f}  (1.0=每簇纯一个数字)")
print("=> 无监督聚类已能把大部分手写数字按真实类别分开")

## 4 · GMM 一维 EM（红酒某特征）

取红酒一个特征，拟合 2 个高斯的混合。E 步算 responsibility，M 步更新 μ/σ/π。

In [ ]:
def gauss(x, mu, var): return np.exp(-(x-mu)**2/(2*var))/np.sqrt(2*np.pi*var)

def gmm1d(x, K=2, n_iter=50, seed=0):
    rng=np.random.default_rng(seed)
    mu = rng.choice(x, K); var = np.full(K, x.var()); pi = np.full(K, 1/K)
    for _ in range(n_iter):
        # E 步
        r = pi*np.array([gauss(x, mu[k], var[k]) for k in range(K)]).T   # (n,K)
        r /= r.sum(1, keepdims=True)
        # M 步
        Nk = r.sum(0)
        pi = Nk/len(x)
        mu = (r*x[:,None]).sum(0)/Nk
        var = (r*(x[:,None]-mu)**2).sum(0)/Nk + 1e-6
    return mu, var, pi, r

x = Xw[:,0]   # alcohol（已标准化）
mu, var, pi, r = gmm1d(x, K=2)
print(f"两个高斯成分: μ={mu.round(2)}  σ={np.sqrt(var).round(2)}  混合权重 π={pi.round(2)}")
print(f"软分配示例（前3个点属于成分0的概率）: {r[:3,0].round(2)}")

## 5 · PCA 的两副面孔：最大方差 = 最小重构误差（Eckart–Young）

讲解里说 PCA 既是「找方差最大的方向」，也是「找重构误差最小的低维子空间」——这两个目标其实是**同一回事**，由 Eckart–Young 定理保证。下面在真实手写数字上把它验证到机器精度：用前 $k$ 个主成分把每张 64 维数字图投影再重构，**实测**的 Frobenius 相对重构误差，应当**精确等于**被丢弃的那些奇异值的能量占比 $\sum_{i>k}\sigma_i^2/\sum_i\sigma_i^2$。两列数字逐行相等，就证明了这个等价。

In [ ]:
# PCA 的「最大方差 = 最小重构误差」（Eckart–Young）：用前 k 个主成分重构手写数字，
# 看重构误差是否精确等于「被丢弃的奇异值的能量」。
Xc = Xd - Xd.mean(0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
total_energy = (S**2).sum()

def reconstruct(k):
    Zk = Xc @ Vt[:k].T            # 投影到前 k 维
    Xhat = Zk @ Vt[:k]            # 再升回 64 维（最优低秩重构）
    return Xhat

print(f"{'k':>3} | {'重构相对误差(实测)':>18} | {'尾部奇异值能量比(理论)':>22}")
for k in [2, 5, 10, 20, 40]:
    Xhat = reconstruct(k)
    rel_err = ((Xc - Xhat)**2).sum() / total_energy       # 实测：‖X-X̂‖²_F / ‖X‖²_F
    tail = (S[k:]**2).sum() / total_energy                 # 理论：Eckart–Young 给出的最优重构误差
    print(f"{k:>3} | {rel_err:>18.6f} | {tail:>22.6f}")
    assert abs(rel_err - tail) < 1e-9, "重构误差应精确等于尾部奇异值能量（Eckart–Young）"

evr10 = (S[:10]**2).sum()/total_energy
print(f"\n=> 前 10 个主成分（占 64 维的 1/6）保留了 {evr10:.1%} 的方差/信息")
print("=> PCA 的两个等价目标在数值上完全吻合：最大化保留方差 ⇔ 最小化 Frobenius 重构误差（Eckart–Young 定理）")

---
## ✏️ 练习区

### ✏️ 练习 1：PCA 与累计解释方差

实现 `pca_evr(X, k)` 返回 `(Z, evr)`：Z 是中心化后投影到前 k 主成分的坐标，evr 是**全部**奇异值的解释方差比向量。

> optdigits 的 64 维信息较分散：前 2 个主成分只解释约 28%，前 10 个约 74%，要累计到 80% 需约 13 个主成分。

In [ ]:
def pca_evr(X, k):
    # TODO: 中心化 -> svd -> evr = S^2/sum(S^2) -> Z = Xc @ Vt[:k].T
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测 ——
Z, evr = pca_evr(Xd, 2)
assert Z.shape == (len(Xd), 2)
assert abs(evr.sum() - 1.0) < 1e-9, "解释方差比应和为1"
assert (evr[:-1] >= evr[1:]-1e-9).all(), "evr 应降序"
assert evr[:10].sum() > 0.70, "前10主成分应解释>70%方差（optdigits 实际约74%）"
# 达到 80% 累计方差需要多少个主成分？optdigits 上约 13 个（远多于 2 个，可见 64 维信息较分散）
k80 = int(np.argmax(np.cumsum(evr) >= 0.80)) + 1
assert 13 <= k80 <= 16, "达到80%累计方差应需约13-16个主成分"
print(f"练习 1 通过 ✓  前2主成分解释 {evr[:2].sum():.1%}，前10解释 {evr[:10].sum():.1%}，达80%需 {k80} 个主成分")


### ✏️ 练习 2：k-means 的 inertia

实现 `kmeans_inertia(X, labels, centers)`：返回簇内平方和。再实现一步 Lloyd 更新 `lloyd_step(X, centers)` 返回新标签与新中心。

In [ ]:
def lloyd_step(X, centers):
    # TODO: 分配到最近中心 -> 中心取均值；返回 (labels, new_centers)
    raise NotImplementedError
def kmeans_inertia(X, labels, centers):
    # TODO: sum ||x - center[label]||^2
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测 ——
rng=np.random.default_rng(0); c0 = Xd[rng.choice(len(Xd),10,replace=False)]
lab1, c1 = lloyd_step(Xd, c0)
i0 = kmeans_inertia(Xd, lab1, c0)
lab2, c2 = lloyd_step(Xd, c1)
i1 = kmeans_inertia(Xd, lab2, c1)
assert i1 <= i0 + 1e-6, "Lloyd 每步 inertia 不增"
print(f"练习 2 通过 ✓  inertia {i0:,.0f} -> {i1:,.0f}（单调下降）")


### ✏️ 练习 3：cluster purity

实现 `cluster_purity(labels, y_true)`。然后验证：k-means 在标准化红酒上的 purity 应该相当高。

In [ ]:
def cluster_purity(labels, y_true):
    # TODO: 每个簇取众数真实标签，累加正确数 / 总数
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
lab,_,_ = kmeans(Xw, 3, seed=0)
pur = cluster_purity(lab, yw)
assert 0 <= pur <= 1
assert pur > 0.85, "标准化红酒 3 簇 purity 应 >0.85"
# 随机标签 purity 应接近最大类占比
rand = np.random.default_rng(0).integers(0,3,len(yw))
assert cluster_purity(rand, yw) < pur
print(f"练习 3 通过 ✓  红酒聚类 purity={pur:.3f}")


### ✏️ 练习 4：GMM E 步（responsibilities）

实现 `e_step(x, mu, var, pi)` 返回 responsibility 矩阵 `(n,K)`，每行和为 1。

In [ ]:
def e_step(x, mu, var, pi):
    # TODO: r[i,k] ∝ pi[k]*N(x_i|mu_k,var_k)，按行归一化
    raise NotImplementedError


In [ ]:
# —— 练习 4 自测 ——
x = Xw[:,0]
r = e_step(x, np.array([-1.,1.]), np.array([1.,1.]), np.array([0.5,0.5]))
assert r.shape == (len(x), 2)
assert np.allclose(r.sum(1), 1.0), "每行 responsibility 和为1"
# 离 mu0 更近的点应更属于成分0
assert r[x.argmin(),0] > 0.5
print("练习 4 通过 ✓")


---
## 📖 参考答案

In [ ]:
# 练习 1
def pca_evr(X, k):
    Xc = X - X.mean(0); U,S,Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:k].T, S**2/(S**2).sum()
print("练习 1 ✓")

In [ ]:
# 练习 2
def lloyd_step(X, centers):
    d = ((X[:,None,:]-centers[None,:,:])**2).sum(2); lab = d.argmin(1)
    new = np.array([X[lab==j].mean(0) if (lab==j).any() else centers[j] for j in range(len(centers))])
    return lab, new
def kmeans_inertia(X, labels, centers):
    return float(((X - centers[labels])**2).sum())
print("练习 2 ✓")

In [ ]:
# 练习 3
def cluster_purity(labels, y_true):
    return sum(np.bincount(y_true[labels==c]).max() for c in np.unique(labels))/len(y_true)
print("练习 3 ✓")

In [ ]:
# 练习 4
def e_step(x, mu, var, pi):
    comp = np.array([pi[k]*np.exp(-(x-mu[k])**2/(2*var[k]))/np.sqrt(2*np.pi*var[k]) for k in range(len(mu))]).T
    return comp/comp.sum(1, keepdims=True)
print("练习 4 ✓ —— E 步给软分配，M 步用它更新参数，交替即 EM")